In [1]:
import os
import numpy as np
import matplotlib.pyplot as pl

In [3]:
os.chdir('C:\cygwin64\home\codes\dispatch2\data\eos\Tomida+Hori_2016')
import EOS
from scaling import cgs

In [4]:
eos=EOS.eos_t()

reading data from table tomida.dat


In [5]:
d=1e-12
T=100
P=eos.pressure(T,d)
U=eos.U(T,d)
P,U

(0.0035106052664206065, 8461278696.0848522)

In [142]:
pl.figure(1)
pl.clf()
T=np.logspace(2.,3.5)
for d in (1e-14,1e-13,1e-12,1e-11):
    U=eos.U(T,d)
    pl.loglog(T,U)
pl.xlabel('T [K]')
pl.ylabel('U [erg/g]');

To follow an adiabat we need to have $$\Delta U = \frac{P}{\rho}\Delta\ln\rho,$$ where 

$$\Delta U=U(T+\Delta T,\ln\rho+\Delta\ln\rho)-U(T,\ln\rho),$$ 

and since $U$ depends mainly on $T$, this may be approximated by

$$U \frac{\partial \ln U}{\partial \ln T} \Delta\ln T \approx  \frac{P}{\rho}\Delta\ln\rho,$$
or 
$$\frac{\Delta\ln T }{\Delta\ln\rho} \approx  \frac{P}{\rho U\frac{\partial \ln U}{\partial \ln T}},$$


Compute adiabats, with 200 points covering 5 decades in density, starting from 9 different (rho,T) points:

Note that this step may be skipped if it has been executed before; then proceed to the restore!

In [101]:
#adiabats={}
for T0 in [30]:
    for logd_min in [-12,-11,-10]:
        d=np.logspace(logd_min,logd_min+10,501)
        np.log10(d[1]/d[0])
        dlnd=np.log(d[1]/d[0])
        nd=len(d)
        T=np.zeros(nd)
        U=np.zeros(nd)
        P=np.zeros(nd)
        T[0]=T0
        U[0]=eos.U(T[0],d[0])
        P[0]=eos.pressure(T[0],d[0])

        for ir in range(1,nd):
            gamma=eos.gamma(T[ir-1],d[ir-1])
            T[ir]=T[ir-1]*np.exp(dlnd*(gamma-1.0))
            dlnT=1.0
            nit=0
            while(abs(dlnT) > 1e-4):
                U[ir]=eos.U(T[ir],d[ir])
                P[ir]=eos.pressure(T[ir],d[ir])
                d1=(d[ir]+d[ir-1])/2.0
                T1=(T[ir]+T[ir-1])/2.0
                U1=(U[ir]+U[ir-1])/2.0
                P1=(P[ir]+P[ir-1])/2.0
                dUdlnT=U1*eos.dlnUdlnT(T1,d1)
                dU=U[ir]-U[ir-1]
                Pd=(P[ir]+P[ir-1])/(d[ir]+d[ir-1])
                Eps=dU-Pd*dlnd
                dEpsdlnT=dUdlnT-P1/d1*dlnd
                dlnT=-Eps/dEpsdlnT
                T[ir]=T[ir]*np.exp(dlnT)
                nit+=1
            if ir%100==0:
                print(ir)
        ir=nd-1
        key='T{:2d}d-{:02d}'.format(T0,-logd_min)
        print('{} {:2d}  T:{:7.1f}  P:{:11.3e}  nit:{:2d}'.format(
            key,ir,T[ir],P[ir],nit))
        adiabats[key]={'T':T,'P':P,'U':U,'d':d}

100
200
300
400
500
T30d-12 500  T:21782.0  P:  1.409e+10  nit: 2
100
200
300
400
500
T30d-11 500  T:37847.1  P:  2.419e+11  nit: 2
100
200
300
400
500
T30d-10 500  T:67521.4  P:  3.772e+12  nit: 2


In [136]:
adiabats.keys()

dict_keys(['T100d-12', 'T100d-11', 'T100d-10', 'T30d-12', 'T30d-11', 'T30d-10'])

In [137]:
import save_restore as sr
sr.save('Jupiter1.save',adiabats)

Restore from this point when restarting

In [138]:
import save_restore as sr
adiabats=sr.restore('Jupiter1.save')
adiabats.keys()

dict_keys(['T100d-12', 'T100d-11', 'T100d-10', 'T30d-12', 'T30d-11', 'T30d-10'])

In [141]:
label='Jupiter1'
pl.figure(1); pl.clf(); pl.title('adiabats')
pl.xlabel('d [cgs]'); pl.ylabel('T [K]')
for k,v in adiabats.items():
    if k[0:3]=='T10':
        i0=v['i0']
        pl.loglog(v['d'],v['T'])
        pl.loglog(v['d'][0],v['T'][0],'o')
        pl.loglog(v['d'][i0],v['T'][i0],'o')
pl.savefig(label+'_fig1.png')
pl.figure(2); pl.clf(); pl.title('adiabats')
pl.xlabel('d [cgs]'); pl.ylabel('P [bar]')
for k,v in adiabats.items():
    if k[0:3]=='T10':
        i0=v['i0']
        pl.loglog(v['d'],1e-6*v['P'])
        pl.loglog(v['d'][0],1e-6*v['P'][0],'o')
        pl.loglog(v['d'][i0],1e-6*v['P'][i0],'o')
pl.savefig(label+'_fig2.png')
pl.figure(3); pl.clf(); pl.title('adiabats')
pl.xlabel('d [cgs]'); pl.ylabel('U [erg/g]')
for k,v in adiabats.items():
    if k[0:3]=='T10':
        i0=v['i0']
        pl.loglog(v['d'],v['U'])
        pl.loglog(v['d'][0],v['U'][0],'o')
        pl.loglog(v['d'][i0],v['U'][i0],'o')
pl.savefig(label+'_fig3.png')

Now that we have an adiabat, we can integrate the hydrostatic pressure balance equation, turning $\Delta\ln P$ into $\Delta \ln r$, where $r$ is radius:

$$\frac{dP}{dR}=-g\rho,$$
so
$$\frac{d\ln P}{d\ln R}=-\frac{g\rho R}{P}=-\frac{G M\rho}{R P},$$

$$\frac{d\ln R}{d\ln P}=-\frac{P}{g\rho R}=-\frac{R P}{G M\rho},$$


At small radii, where the density and pressure are nearly constant:
    
$$\frac{dP}{dR}= -\frac{G M(R)\rho}{R^2} \approx -\frac{4\pi G R 
                                                        \rho_0^2}{3},$$

$$\frac{d\ln P}{d R}= -\frac{4\pi G R \rho_0^2}{3 P_0},$$

$$\ln P= \ln P_0 - \frac{2\pi G R^2 \rho_0^2}{3 P_0},$$

So, if $P_0$ is the central pressure and $P_1$ is the next pressure we know 
on the adiabat, the corresponding radius, $R=R_1$, is given by

$$\frac{2\pi G R_1^2 \rho_0^2}{3 P_0} = \ln(P_0/P_1) $$

$$R_1= \left( \frac{3 P_0 \ln(P_0/P_1)}{2\pi G \rho_0^2} \right)^{1/2} $$

The dimension of ${1}/({G\rho_0})$ is time squared, and the dimension of 
$P_0/\rho_0$ is velocity squared, so dimensionally, the result is indeed 
a length.

Make sure that the figures are empty:

In [9]:
from scaling import cgs
for i in (4,5,6):
    pl.figure(i); pl.clf()

Integrate the hydrostatic equation:

In [252]:
#key='T30d-1'
#Me=[472.6,315,200]     # aJ=1.0, T=30
#Me=[600,350,216]       # aJ=5.2, T=30
key='T100d-'
#Me=[1123,613.1,370]    # aJ=1.0, T=100
Me=[1090.3,686,400]    # aJ=5.2, T=100
aJ=5.2
title='Embryo mass: {}'.format(Me)
j=0
for k,v in adiabats.items():
    if k[0:6]==key:
        me=Me[j]
        j+=1
        RE=cgs.r_earth*me**0.333
        ME=cgs.m_earth*me
        P=v['P']
        d=v['d']
        T=v['T']
        nd=len(d)
        R=np.ones(nd)
        M=np.zeros(nd)
        R[0]=aJ*RE*245.
        i=0
        M[i]=ME
        while(i < nd-1 and R[i] > 0.5*cgs.r_earth):
            i=i+1
            R[i]=R[i-1]
            M[i]=M[i-1]
            for nit in range(5):
                F=(R[i]*P[i]/(d[i]*M[i])+R[i-1]*P[i-1]/(d[i-1]*M[i-1]))/2.0
                dlnRdlnP=F/cgs.grav
                dlnP=np.log(P[i-1]/P[i])
                dlnR=dlnP*dlnRdlnP
                R[i]=R[i-1]*np.exp(dlnR)
                dM=4./3.*np.pi*(R[i-1]**3-R[i]**3)*(d[i]+d[i-1])/2.
                M[i]=M[i-1]-dM
        print('    {:2d}  dlnRdlnP:{:9.3f}  R:{:9.3f}  M:{:9.3f}'.format(
            i-1,dlnRdlnP,R[i-1]/cgs.r_earth,M[i-1]/cgs.m_earth))
        #print('{:3d}{:14.3e}{:14.3e}{:14.3e}{:14.3e}{:14.3e}'.format(
        #      i-1,dlnRdlnP,d[i-1],T[i-1],R[i-1]/cgs.r_earth),
        #      M[i-1]/cgs.m_earth)
        if 1:
            print(i-1,d[i-1])
            pl.figure(4)
            pl.xlabel('R/Re'); pl.ylabel('P [bar]')
            pl.title(title)
            pl.loglog(R[0:i]/cgs.r_earth,1e-6*P[0:i],'-+')
            pl.savefig(label+'_fig4.png')
            pl.figure(5)
            pl.xlabel('R/Re'); pl.ylabel('T [K]')
            pl.title(title)
            pl.loglog(R[0:i]/cgs.r_earth,T[0:i])
            pl.savefig(label+'_fig5.png')
            pl.figure(6)
            pl.xlabel('R/Re'); pl.ylabel('M [M_E]')
            pl.title(title)
            pl.loglog(R[0:i]/cgs.r_earth,M[0:i]/cgs.m_earth,'-+')
            pl.savefig(label+'_fig6.png')
        
        v['M']=M
        v['R']=R
        v['i0']=i
        adiabats[k]=v

    857  dlnRdlnP:   76.609  R:    6.978  M:    0.053
857 0.000371535229097
    789  dlnRdlnP:  196.495  R:    2.301  M:    0.005
789 0.000776247116629
    999  dlnRdlnP:    0.351  R:    0.644  M:   28.050
999 0.977237220956


In [253]:
sr.save('Jupiter1.save',adiabats)

In [250]:
v.keys()

dict_keys(['T', 'P', 'U', 'd', 'M', 'R', 'i0'])

Construct interpolators for pressure along adiabat $P(\rho)$:

In [171]:
from scipy import interpolate
i0=v['i0']
print(len(d),len(P),i0)
pressure=interpolate.interp1d(d[0:i0+1],P[0:i0+1])
pl.figure(10)
pl.loglog(d[0:i0+1],P[0:i0+1],'o');
d1=np.logspace(np.log10(d[i0-10]),np.log10(d[i0]))
P1=pressure(d1)
pl.loglog(d1,P1,'+')

501 501 443


Verify that the structure is hydrostatic ($d\ln P=\frac{G M\rho}{R P}d\ln R$)

In [93]:
pl.figure(7)
pl.clf()
i0=v['i0']
R=v['R'][0:i0]
P=v['P'][0:i0]
T=v['T'][0:i0]
M=v['M'][0:i0]
d=v['d'][0:i0]
dlnP=np.gradient(np.log(P))
dlnR=np.gradient(np.log(R))
g=cgs.grav*M/R**2
dlnPdlnR=-g*d*R/P
pl.semilogx(R/cgs.r_earth,dlnP,'-+')
pl.semilogx(R/cgs.r_earth,dlnPdlnR*dlnR,'-+')
pl.savefig(label+'_fig7.png')

In [96]:
pl.figure(7)
pl.clf()
i0=v['i0']
R=v['R'][0:i0]
P=v['P'][0:i0]
T=v['T'][0:i0]
M=v['M'][0:i0]
d=v['d'][0:i0]
dlnP=np.gradient(np.log(P))
dlnR=np.gradient(np.log(R))
dR=np.gradient(R)
g=cgs.grav*M/R**2
dlnPdlnR=-g*d*R/P
pl.semilogx(R/cgs.r_earth,dlnP,'-+')
pl.semilogx(R/cgs.r_earth,-g*d/P*dR,'-+')
pl.savefig(label+'_fig7.png')

Verify that entropy is constant ($dU=\frac{P}{\rho}d\ln\rho$)

In [86]:
pl.figure(8)
pl.clf()
U=eos.U(T,d)
dU=np.gradient(U)
dlnd=np.gradient(np.log(d))
pl.semilogx(R/cgs.r_earth,dU)
pl.semilogx(R/cgs.r_earth,dlnd*P/d,'+')
pl.savefig(label+'_fig8.png')